In [1]:
import torch
import torch.nn.functional as F

from datasets import load_from_disk
from pathlib import Path
from typing import Dict

from config import Config
from data_utils import generate_causal_mask
from pipeline import get_dataloaders
from transformer.transformer_model import TransformerNMT, TransformerConfig
from tqdm import tqdm


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset_path = Path("parallel_en_fr_corpus")
batch_size = 32
epochs = 10
lr = 1e-3
save_dir = "checkpoints"

cfg = TransformerConfig(
    d_model=32,
    n_heads=4,
    n_encoder_layers=3,
    n_decoder_layers=3,
    d_ff=32 * 4,
    max_seq_len=32,
    dropout=0.1,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
ds = load_from_disk(dataset_path)
train_fr = ds["train"]["text_fr"]
train_en = ds["train"]["text_en"]
val_fr = ds["validation"]["text_fr"]
val_en = ds["validation"]["text_en"]
test_fr = ds["test"]["text_fr"]
test_en = ds["test"]["text_en"]

train_loader, val_loader, test_loader, fr_tok, en_tok = get_dataloaders(
    train_fr=train_fr,
    train_en=train_en,
    val_fr=val_fr,
    val_en=val_en,
    test_fr=test_fr,
    test_en=test_en,
    batch_size=batch_size,
)

## Training the Transformer NMT Model

In [6]:
def train_one_epoch(
    model: TransformerNMT,
    loader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)
        src_padding_mask = batch["src_padding_mask"].to(device)

        # Teacher forcing: decoder input excludes last token, labels exclude first token.
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        tgt_padding_mask = (tgt_in != Config.PAD_TOKEN_ID)
        causal_mask = generate_causal_mask(tgt_in.size(1)).to(device)

        logits, _ = model(
            src,
            tgt_in,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            causal_mask=causal_mask,
        )

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1),
            ignore_index=Config.PAD_TOKEN_ID,
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            non_pad = (tgt_out != Config.PAD_TOKEN_ID).sum().item()
            total_tokens += int(non_pad)
            total_loss += float(loss.item()) * max(1, int(non_pad))

    return total_loss / max(1, total_tokens)


In [7]:
@torch.no_grad()
def eval_one_epoch(
    model: TransformerNMT,
    loader,
    device: torch.device,
) -> float:
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)
        src_padding_mask = batch["src_padding_mask"].to(device)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        tgt_padding_mask = (tgt_in != Config.PAD_TOKEN_ID)
        causal_mask = generate_causal_mask(tgt_in.size(1)).to(device)

        logits, _ = model(
            src,
            tgt_in,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            causal_mask=causal_mask,
        )

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1),
            ignore_index=Config.PAD_TOKEN_ID,
        )

        non_pad = (tgt_out != Config.PAD_TOKEN_ID).sum().item()
        total_tokens += int(non_pad)
        total_loss += float(loss.item()) * max(1, int(non_pad))

    return total_loss / max(1, total_tokens)

In [8]:
def save_checkpoint(path: Path, model: TransformerNMT, optimizer: torch.optim.Optimizer, meta: Dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "meta": meta,
        },
        str(path),
    )

In [ ]:
model = TransformerNMT(
    src_vocab_size=len(fr_tok),
    tgt_vocab_size=len(en_tok),
    cfg=cfg,
    pad_token_id=Config.PAD_TOKEN_ID,
    bos_token_id=Config.BOS_TOKEN_ID,
    eos_token_id=Config.EOS_TOKEN_ID,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = eval_one_epoch(model, val_loader, device)

    print(f"Epoch {epoch:02d}/{epochs} | train loss/token: {train_loss:.4f} | val loss/token: {val_loss:.4f}")

    ckpt_path = Path(save_dir) / f"transformer_epoch_{epoch:02d}.pt"
    save_checkpoint(
        ckpt_path,
        model,
        optimizer,
        meta={"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "cfg": cfg.__dict__},
    )

Epoch 01/10 | train loss/token: 5.3311 | val loss/token: 3.1253
Epoch 02/10 | train loss/token: 2.9295 | val loss/token: 2.5683
Epoch 03/10 | train loss/token: 2.5222 | val loss/token: 2.3314
Epoch 04/10 | train loss/token: 2.3306 | val loss/token: 2.2120
Epoch 05/10 | train loss/token: 2.2039 | val loss/token: 2.1268
Epoch 06/10 | train loss/token: 2.1088 | val loss/token: 2.0641
Epoch 07/10 | train loss/token: 2.0214 | val loss/token: 2.0050
Epoch 08/10 | train loss/token: 1.9406 | val loss/token: 1.9430
Epoch 09/10 | train loss/token: 1.8676 | val loss/token: 1.9016
Epoch 10/10 | train loss/token: 1.7946 | val loss/token: 1.8565
